In [ ]:
from pathlib import Path

from src.data.dashboard import open_portfolio
from src.models.explain import ChurnExplainer, waterfall_figure
from src.models.pipeline import FEATURE_COLUMNS
from src.models.predict import ChurnPredictor

predictor = ChurnPredictor.from_final_test(
    Path("mlruns"),
    final_test_run_id="5818be9683af42a485ffc2896a868137",
)
explainer = ChurnExplainer(predictor)

with open_portfolio(Path(r"data\processed\portfolio_v1")) as connection:
    training = connection.sql(
        "SELECT * FROM portfolio WHERE partition = 'train'"
    ).df()

inputs = training.loc[:, ["customer_id", *FEATURE_COLUMNS]]

# Local explanation
local = explainer.explain_one(inputs.iloc[0].to_dict())
display(local.contributions)
waterfall_figure(local, top_n=10).show()

# Global importance: deterministic, bounded training sample
global_result = explainer.global_importance(
    training.loc[:, ["customer_id", "partition", *FEATURE_COLUMNS]],
    sample_size=1000,
)
display(global_result.importance)

ModuleNotFoundError: No module named 'src'